# 1\.1\.5 Load Subway Ridership

This notebook stages, validates, and profiles the MTA Subway Hourly Ridership datasets covering Q1 2023 through Q1 2026\. The source data is obtained directly from the New York State Open Data platform via the Socrata API and includes hourly ridership and transfer activity by station complex, payment method, fare class, and transit mode\. 

The primary objective of this notebook is to transform the raw quarterly extracts into a consistent parquet\-based analytical dataset suitable for downstream harmonization with the other mobility systems included in this project\. In addition to source extraction and parquet conversion, the notebook performs a series of quality assurance checks covering schema consistency, temporal coverage, missingness, candidate\-key uniqueness, metric validity, station coverage, geographic completeness, and categorical field distributions\. 

The resulting staged Subway dataset provides approximately 88 million hourly observations spanning the NYC Subway system, Staten Island Railway, and Roosevelt Island Tram\. These data will later be aligned with Taxi Zones and integrated with roadway traffic, taxi/FHV, bus, and bridge/tunnel datasets to support multimodal analysis of congestion pricing impacts across the New York City transportation network\. ::

In [1]:
# ---------------------------------------------------------------------
# Imports
# ---------------------------------------------------------------------

from pathlib import Path
import json
import time

import requests
import pandas as pd
import duckdb

## 1\.1\.5\.1 Load Subway Hourly Ridership

In this section, we load and stage the MTA Subway Hourly Ridership datasets for the project period\. Subway ridership will become one of the core mobility signals in the final Taxi Zone × Date × Temporal Bucket modeling table, alongside traffic volumes, bus speeds, and Taxi/FHV activity\.

The subway source data is split into two Socrata datasets:

- MTA Subway Hourly Ridership: 2020–2024

- MTA Subway Hourly Ridership: Beginning 2025

For this project, we only need the 2023–2024 portion of the historical dataset, plus the 2025\+ dataset for the post\-congestion\-pricing period\. We will follow the same staged, parquet\-first pattern used for the Bus Speeds dataset: download the raw files into source\_data/, inspect schemas and row counts, convert the data into pipeline parquet outputs, and create lightweight QA manifests so expensive scans do not need to be repeated unnecessarily\.

In [2]:
# ---------------------------------------------------------------------
# Download MTA subway hourly ridership data by quarter via Socrata API
# ---------------------------------------------------------------------

RUN_SUBWAY_DOWNLOAD = False

SOURCE_DATA_DIR = Path("source_data")
SUBWAY_DATA_DIR = SOURCE_DATA_DIR / "subway"
SUBWAY_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Socrata downloads are paged because the full export is large and can fail midstream.
# Keeping pages at the quarter level makes the download restartable without losing prior work.
SUBWAY_PAGE_LIMIT = 250000

# MTA publishes this series in two Socrata datasets: one historical dataset through 2024
# and a second beginning in 2025. The quarter map keeps the project window explicit.
subway_quarter_downloads = {
    "mta_subway_hourly_ridership_2023_q1": ("wujg-7c2s", "2023-01-01", "2023-04-01"),
    "mta_subway_hourly_ridership_2023_q2": ("wujg-7c2s", "2023-04-01", "2023-07-01"),
    "mta_subway_hourly_ridership_2023_q3": ("wujg-7c2s", "2023-07-01", "2023-10-01"),
    "mta_subway_hourly_ridership_2023_q4": ("wujg-7c2s", "2023-10-01", "2024-01-01"),
    "mta_subway_hourly_ridership_2024_q1": ("wujg-7c2s", "2024-01-01", "2024-04-01"),
    "mta_subway_hourly_ridership_2024_q2": ("wujg-7c2s", "2024-04-01", "2024-07-01"),
    "mta_subway_hourly_ridership_2024_q3": ("wujg-7c2s", "2024-07-01", "2024-10-01"),
    "mta_subway_hourly_ridership_2024_q4": ("wujg-7c2s", "2024-10-01", "2025-01-01"),
    "mta_subway_hourly_ridership_2025_q1": ("5wq4-mkjj", "2025-01-01", "2025-04-01"),
    "mta_subway_hourly_ridership_2025_q2": ("5wq4-mkjj", "2025-04-01", "2025-07-01"),
    "mta_subway_hourly_ridership_2025_q3": ("5wq4-mkjj", "2025-07-01", "2025-10-01"),
    "mta_subway_hourly_ridership_2025_q4": ("5wq4-mkjj", "2025-10-01", "2026-01-01"),
    "mta_subway_hourly_ridership_2026_q1": ("5wq4-mkjj", "2026-01-01", "2026-04-01"),
}

subway_existing_files = sorted(
    SUBWAY_DATA_DIR.glob("mta_subway_hourly_ridership_*_page_*.csv")
)

should_run_subway_download = (
    RUN_SUBWAY_DOWNLOAD
    or len(subway_existing_files) == 0
)

if should_run_subway_download:
    print("Running Subway download.")

    for dataset_key, (socrata_id, start_date, end_date) in subway_quarter_downloads.items():
        quarter_dir = SUBWAY_DATA_DIR / dataset_key
        quarter_dir.mkdir(parents=True, exist_ok=True)

        # A _COMPLETE marker means every page for that quarter was downloaded.
        # If the notebook stops mid-quarter, the next run resumes from the saved pages.
        complete_marker_path = quarter_dir / "_COMPLETE"

        if complete_marker_path.exists():
            print(f"Already complete, skipping: {dataset_key}")
            continue

        print(f"Downloading {dataset_key}...")

        url = f"https://data.ny.gov/resource/{socrata_id}.csv"
        offset = 0
        total_rows = 0

        while True:
            page_number = offset // SUBWAY_PAGE_LIMIT
            page_path = quarter_dir / f"{dataset_key}_page_{page_number:05d}.csv"

            # Existing non-empty pages are trusted and counted, so reruns continue
            # from the first missing page instead of starting the quarter over.
            if page_path.exists() and page_path.stat().st_size > 0:
                rows_this_page = sum(1 for _ in open(page_path, encoding="utf-8")) - 1
                total_rows += rows_this_page

                print(
                    f"{dataset_key}: already have page {page_number:05d} "
                    f"({total_rows:,} rows through this page)"
                )

                if rows_this_page < SUBWAY_PAGE_LIMIT:
                    complete_marker_path.touch()
                    break

                offset += SUBWAY_PAGE_LIMIT
                continue

            response = requests.get(
                url,
                params={
                    "$where": (
                        f"transit_timestamp >= '{start_date}T00:00:00' "
                        f"AND transit_timestamp < '{end_date}T00:00:00'"
                    ),
                    "$order": (
                        "transit_timestamp, "
                        "station_complex_id, "
                        "payment_method, "
                        "fare_class_category"
                    ),
                    "$limit": SUBWAY_PAGE_LIMIT,
                    "$offset": offset,
                },
                timeout=300,
            )

            response.raise_for_status()

            lines = response.text.splitlines()

            if len(lines) <= 1:
                complete_marker_path.touch()
                break

            with open(page_path, "w", encoding="utf-8", newline="") as f:
                f.write("\n".join(lines))
                f.write("\n")

            rows_this_page = len(lines) - 1
            total_rows += rows_this_page

            print(
                f"{dataset_key}: downloaded page {page_number:05d} "
                f"({total_rows:,} rows)"
            )

            if rows_this_page < SUBWAY_PAGE_LIMIT:
                complete_marker_path.touch()
                break

            offset += SUBWAY_PAGE_LIMIT

else:
    print(
        "Skipping Subway download. Files already exist. "
        "Set RUN_SUBWAY_DOWNLOAD = True to force rerun."
    )

Running Subway download.
Already complete, skipping: mta_subway_hourly_ridership_2023_q1
Already complete, skipping: mta_subway_hourly_ridership_2023_q2
Already complete, skipping: mta_subway_hourly_ridership_2023_q3
Already complete, skipping: mta_subway_hourly_ridership_2023_q4
Already complete, skipping: mta_subway_hourly_ridership_2024_q1
Already complete, skipping: mta_subway_hourly_ridership_2024_q2
Already complete, skipping: mta_subway_hourly_ridership_2024_q3
Already complete, skipping: mta_subway_hourly_ridership_2024_q4
Already complete, skipping: mta_subway_hourly_ridership_2025_q1
Already complete, skipping: mta_subway_hourly_ridership_2025_q2
Already complete, skipping: mta_subway_hourly_ridership_2025_q3
Already complete, skipping: mta_subway_hourly_ridership_2025_q4
Already complete, skipping: mta_subway_hourly_ridership_2026_q1


The Subway Hourly Ridership download is complete\. Because the Socrata full\-export endpoint was unstable for this dataset, we downloaded the source data by quarter instead of as one large file\. This gives us smaller, restartable source files while still preserving the raw dataset structure for the 1\.1\.5 load step\.

### Lightweight CSV Validation

Let's do a lightweight validation pass to confirm that each downloaded CSV can be read and has a reasonable row count\. The file sizes look consistent, but this check gives us more confidence that none of the quarterly downloads were truncated or malformed\.

In [3]:
# ---------------------------------------------------------------------
# Validate downloaded Subway source folders with a QA manifest
# ---------------------------------------------------------------------

SUBWAY_QA_MANIFEST_PATH = SUBWAY_DATA_DIR / "subway_source_file_manifest.json"

subway_source_dirs = sorted(
    path
    for path in SUBWAY_DATA_DIR.glob("mta_subway_hourly_ridership_20*_q*")
    if path.is_dir()
)

print(f"Found {len(subway_source_dirs)} Subway source folders.")

if SUBWAY_QA_MANIFEST_PATH.exists():
    with open(SUBWAY_QA_MANIFEST_PATH, "r") as f:
        subway_qa_manifest = json.load(f)
else:
    subway_qa_manifest = {}

for source_dir in subway_source_dirs:
    file_key = source_dir.name
    page_files = sorted(source_dir.glob("*.csv"))

    if len(page_files) == 0:
        print(f"No page files found, skipping: {file_key}")
        continue

    size_bytes = sum(
        path.stat().st_size
        for path in page_files
    )

    modified_time = max(
        path.stat().st_mtime
        for path in page_files
    )

    existing_record = subway_qa_manifest.get(file_key)

    # The manifest prevents repeated full scans of already-validated source folders.
    # If page count, total size, and modified time match, the cached row count is still usable.
    if (
        existing_record is not None
        and existing_record["num_page_files"] == len(page_files)
        and existing_record["size_bytes"] == size_bytes
        and existing_record["modified_time"] == modified_time
    ):
        print(
            f"Already validated, skipping: "
            f"{file_key} ({existing_record['row_count']:,} rows)"
        )
        continue

    print(f"Validating: {file_key}")

    row_count = 0

    # Count rows page by page so malformed or truncated CSV pages surface before parquet staging.
    for page_path in page_files:
        page_row_count = duckdb.sql(f"""
            SELECT COUNT(*) AS row_count
            FROM read_csv_auto('{page_path}')
        """).fetchone()[0]

        row_count += page_row_count

    subway_qa_manifest[file_key] = {
        "num_page_files": len(page_files),
        "size_bytes": size_bytes,
        "size_mb": size_bytes / (1024 * 1024),
        "modified_time": modified_time,
        "row_count": row_count,
    }

    with open(SUBWAY_QA_MANIFEST_PATH, "w") as f:
        json.dump(subway_qa_manifest, f, indent=2)

    print(
        f"{file_key}: "
        f"{len(page_files)} page files, "
        f"{size_bytes / (1024 * 1024):.1f} MB, "
        f"{row_count:,} rows"
    )

Found 13 Subway source folders.
Already validated, skipping: mta_subway_hourly_ridership_2023_q1 (6,268,873 rows)
Already validated, skipping: mta_subway_hourly_ridership_2023_q2 (6,413,089 rows)
Already validated, skipping: mta_subway_hourly_ridership_2023_q3 (6,375,366 rows)
Already validated, skipping: mta_subway_hourly_ridership_2023_q4 (6,528,085 rows)
Already validated, skipping: mta_subway_hourly_ridership_2024_q1 (6,445,692 rows)
Already validated, skipping: mta_subway_hourly_ridership_2024_q2 (6,558,471 rows)
Already validated, skipping: mta_subway_hourly_ridership_2024_q3 (6,692,309 rows)
Already validated, skipping: mta_subway_hourly_ridership_2024_q4 (7,316,041 rows)
Already validated, skipping: mta_subway_hourly_ridership_2025_q1 (7,271,588 rows)
Already validated, skipping: mta_subway_hourly_ridership_2025_q2 (8,095,749 rows)
Already validated, skipping: mta_subway_hourly_ridership_2025_q3 (7,974,913 rows)
Already validated, skipping: mta_subway_hourly_ridership_2025_q4 (

Findings: Subway extraction completed successfully across 13 quarterly folders spanning Q1 2023 through Q1 2026\. Quarterly row counts range from approximately 5\.3 million to 8\.1 million records, with a noticeable increase in volume beginning in late 2024 and throughout 2025\. The staged source extracts contain roughly 88 million records in total and appear complete based on the expected page counts and file sizes\.

### Lightweight Orientation

Before converting the Subway source files into parquet, let's perform a lightweight schema check across the quarterly Subway extracts to confirm that all downloaded files share a consistent column structure\.

In [4]:
# ---------------------------------------------------------------------
# Lightweight orientation of downloaded Subway source folders
# ---------------------------------------------------------------------

subway_source_dirs = sorted(
    path
    for path in SUBWAY_DATA_DIR.glob("mta_subway_hourly_ridership_20*_q*")
    if path.is_dir()
)

subway_column_checks = {}

for source_dir in subway_source_dirs:
    page_files = sorted(source_dir.glob("*.csv"))

    if len(page_files) == 0:
        print(f"No page files found, skipping: {source_dir.name}")
        continue

    sample = pd.read_csv(
        page_files[0],
        nrows=100
    )

    subway_column_checks[source_dir.name] = sample.columns.tolist()

    del sample

reference_file = list(subway_column_checks.keys())[0]
reference_columns = subway_column_checks[reference_file]

print("Reference source folder")
print(reference_file)

print("Reference columns")
print(reference_columns)

print("Column consistency")
print(
    all(
        columns == reference_columns
        for columns in subway_column_checks.values()
    )
)

Reference source folder
mta_subway_hourly_ridership_2023_q1
Reference columns
['transit_timestamp', 'transit_mode', 'station_complex_id', 'station_complex', 'borough', 'payment_method', 'fare_class_category', 'ridership', 'transfers', 'latitude', 'longitude', 'georeference']
Column consistency
True


Findings: Schema was fully consistent across all quarterly Subway extracts\. All files contained the expected temporal, station, fare, ridership, transfer, and geographic fields, with no evidence of schema drift between quarters\. This provides a stable foundation for parquet conversion and downstream harmonization\.

Data Dictionary

Transit Timestamp\. Timestamp associated with the ridership observation\. This field represents the primary temporal dimension of the dataset and will be used to derive dates, hours, days of week, temporal buckets, and congestion\-pricing periods\.
Transit Mode\. Transit mode associated with the observation\. Expected to be Subway for all records in this dataset\.
Station Complex ID\. Unique identifier for the subway station complex where the rider entered the system\. More reliable than station names for joins and aggregation\.
Station Complex\. Human\-readable name of the subway station complex associated with the observation\.
Borough\. NYC borough associated with the station complex location\.
Payment Method\. Fare payment method used for the ride, such as OMNY or MetroCard\.
Fare Class Category\. Fare classification associated with the payment transaction, such as full fare, reduced fare, student fare, or other rider categories\.
Ridership\. Estimated number of riders associated with the station complex, payment method, and fare class combination during the observation period\. This serves as the primary ridership metric in the dataset\.
Transfers\. Estimated number of rider transfers associated with the observation\. Transfers are already included within ridership totals and should not be added to ridership counts\.
Latitude\. Latitude coordinate of the subway station complex\.
Longitude\. Longitude coordinate of the subway station complex\.
Georeference\. Geospatial POINT representation of the station complex location\. Provides an alternative spatial representation of the latitude and longitude fields\.

##  1\.1\.5\.2 Convert raw Subway \.csv files into staged parquet files

Although the Subway datasets are much smaller than Bus and TLC, we still convert them into staged parquet files so downstream QA and transformation steps can work from a faster, analytics\-friendly format\.

In [5]:
# ---------------------------------------------------------------------
# Set up pipeline output directory
# ---------------------------------------------------------------------

PIPELINE_DATA_DIR = Path("pipeline_data")

PIPELINE_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [6]:
# ---------------------------------------------------------------------
# 1.1.5.2 Convert raw Subway .csv page files into staged parquet files
# ---------------------------------------------------------------------

RUN_SUBWAY_PARQUET_CONVERSION = False

SUBWAY_PARQUET_DIR = PIPELINE_DATA_DIR / "1.1.5.subway_hourly_ridership_parquet"

SUBWAY_PARQUET_DIR.mkdir(
    parents=True,
    exist_ok=True
)

subway_source_dirs = sorted(
    path
    for path in SUBWAY_DATA_DIR.glob("mta_subway_hourly_ridership_20*_q*")
    if path.is_dir()
)

# Each quarter folder contains multiple downloaded CSV pages.
# This mapping collapses those pages into one staged parquet file per quarter.
subway_dirs_to_parquet = {
    source_dir.name.replace("mta_subway_hourly_ridership_", ""): {
        "source_dir": source_dir,
        "parquet_path": (
            SUBWAY_PARQUET_DIR /
            source_dir.name.replace(
                "mta_subway_hourly_ridership_",
                "subway_"
            )
        ).with_suffix(".parquet"),
    }
    for source_dir in subway_source_dirs
}

subway_existing_parquet_files = sorted(
    SUBWAY_PARQUET_DIR.glob("subway_*.parquet")
)

should_run_subway_parquet_conversion = (
    RUN_SUBWAY_PARQUET_CONVERSION
    or len(subway_existing_parquet_files) == 0
)

if should_run_subway_parquet_conversion:
    print("Running Subway parquet conversion.")

    for dataset_name, paths in subway_dirs_to_parquet.items():
        source_dir = paths["source_dir"]
        parquet_path = paths["parquet_path"]

        if parquet_path.exists() and parquet_path.stat().st_size > 0:
            size_mb = parquet_path.stat().st_size / (1024 * 1024)
            print(f"Already converted, skipping: {parquet_path.name} ({size_mb:.1f} MB)")
            continue

        page_files = sorted(source_dir.glob("*.csv"))

        if len(page_files) == 0:
            print(f"No page files found, skipping: {dataset_name}")
            continue

        print(f"Converting {dataset_name} to parquet...")

        page_file_paths = [
            str(path)
            for path in page_files
        ]

        # DuckDB reads all page files for the quarter and writes a compact parquet artifact.
        # union_by_name keeps the conversion resilient if column order differs across pages.
        duckdb.sql(f"""
            COPY (
                SELECT *
                FROM read_csv_auto(
                    {page_file_paths},
                    union_by_name = true
                )
            )
            TO '{parquet_path}'
            (FORMAT PARQUET);
        """)

        size_mb = parquet_path.stat().st_size / (1024 * 1024)
        print(f"Saved parquet file to: {parquet_path} ({size_mb:.1f} MB)")

else:
    print(
        "Skipping Subway parquet conversion. Files already exist. "
        "Set RUN_SUBWAY_PARQUET_CONVERSION = True to force rerun."
    )

Skipping Subway parquet conversion. Files already exist. Set RUN_SUBWAY_PARQUET_CONVERSION = True to force rerun.


Now that we've staged the Subway source files into parquet format, let's do an initial health check to confirm that each parquet file is readable and contains a reasonable number of records\. Similar to the Taxi/FHV pipeline, this summary provides a quick view of file sizes, row counts, and any conversion failures before we move on to more detailed schema and data\-quality validation\.

In [7]:
# ---------------------------------------------------------------------
# Check Subway parquet health
# ---------------------------------------------------------------------

subway_parquet_files = sorted(
    SUBWAY_PARQUET_DIR.glob("subway_*.parquet")
)

parquet_health_records = []

for path in subway_parquet_files:
    size_mb = path.stat().st_size / (1024 * 1024)

    try:
        num_rows = duckdb.sql(f"""
            SELECT COUNT(*) AS num_rows
            FROM read_parquet('{path}')
        """).fetchone()[0]

        error = None

    except Exception as e:
        num_rows = None
        error = str(e)

    parquet_health_records.append({
        "file": path.name,
        "size_mb": size_mb,
        "num_rows": num_rows,
        "error": error,
    })

parquet_health_df = pd.DataFrame(parquet_health_records)

parquet_health_df

,file,size_mb,num_rows,error
0,subway_2023_q1.parquet,26.392543,6268873,None
1,subway_2023_q2.parquet,27.202653,6413089,None
2,subway_2023_q3.parquet,27.479550,6375366,None
3,subway_2023_q4.parquet,27.564980,6528085,None
4,subway_2024_q1.parquet,27.406576,6445692,None
5,subway_2024_q2.parquet,27.839907,6558471,None
6,subway_2024_q3.parquet,28.890858,6692309,None
7,subway_2024_q4.parquet,30.253305,7316041,None
8,subway_2025_q1.parquet,28.411888,7271588,None
9,subway_2025_q2.parquet,30.238503,8095749,None


In [8]:
# ---------------------------------------------------------------------
# Summarize Subway parquet health
# ---------------------------------------------------------------------

parquet_health_df = parquet_health_df.copy()

parquet_health_df["year"] = parquet_health_df["file"].str.extract(
    r"subway_(\d{4})_q\d"
)

parquet_health_df["quarter"] = parquet_health_df["file"].str.extract(
    r"subway_\d{4}_(q\d)"
)

parquet_health_df["period"] = (
    parquet_health_df["year"] + "_" + parquet_health_df["quarter"]
)

parquet_health_df["readable"] = parquet_health_df["error"].isna()

parquet_health_summary = pd.DataFrame([{
    "num_files": parquet_health_df["file"].count(),
    "readable_files": parquet_health_df["readable"].sum(),
    "unreadable_files": (~parquet_health_df["readable"]).sum(),
    "total_rows": parquet_health_df["num_rows"].sum(),
    "total_size_mb": parquet_health_df["size_mb"].sum(),
    "min_size_mb": parquet_health_df["size_mb"].min(),
    "max_size_mb": parquet_health_df["size_mb"].max(),
}])

parquet_health_summary

,num_files,readable_files,unreadable_files,total_rows,total_size_mb,min_size_mb,max_size_mb
0,13,13,0,88319707,369.980789,26.392543,31.078279


Findings:  The parquet staging step completed successfully\. All 13 Subway parquet files are readable, with zero failed files\. Together, the staged files contain 88,319,707 rows and compress down to about 370 MB, with individual parquet files ranging from roughly 26 MB to 31 MB\.

## 1\.1\.5\.3 Schema QA

Let's validate that the schema remains consistent across all quarterly Subway extracts\. We want to confirm that the expected ridership, fare, station, temporal, and geographic fields are present throughout the dataset and that no unexpected schema drift was introduced during extraction or staging\.

Let's read the schema from each staged Subway parquet file\. This gives us a file\-level view of column names and inferred data types without loading the full dataset into memory\.

In [9]:
# ---------------------------------------------------------------------
# 1.1.5.3 Schema QA for staged Subway parquet files
# ---------------------------------------------------------------------

subway_parquet_files = sorted(
    SUBWAY_PARQUET_DIR.glob("subway_*.parquet")
)

subway_schema_records = []

for path in subway_parquet_files:
    schema_df = duckdb.sql(f"""
        DESCRIBE SELECT *
        FROM read_parquet('{path}')
    """).df()

    for _, row in schema_df.iterrows():
        subway_schema_records.append({
            "file": path.name,
            "column_name": row["column_name"],
            "column_type": row["column_type"],
        })

subway_schema_df = pd.DataFrame(subway_schema_records)

subway_schema_df.head()

,file,column_name,column_type
0,subway_2023_q1.parquet,transit_timestamp,TIMESTAMP
1,subway_2023_q1.parquet,transit_mode,VARCHAR
2,subway_2023_q1.parquet,station_complex_id,VARCHAR
3,subway_2023_q1.parquet,station_complex,VARCHAR
4,subway_2023_q1.parquet,borough,VARCHAR


Summarize how often each column/type combination appears across the staged parquet files\. This helps us quickly spot type inconsistencies or columns that only appear in some periods\.

In [10]:
# ---------------------------------------------------------------------
# Compare Subway schemas across parquet files
# ---------------------------------------------------------------------

subway_schema_summary = (
    subway_schema_df
    .groupby(["column_name", "column_type"], dropna=False)
    .agg(
        num_files=("file", "nunique"),
        files=("file", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values(["column_name", "column_type"])
)

subway_schema_summary

,column_name,column_type,num_files,files
0,borough,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
1,fare_class_category,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
2,georeference,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
3,latitude,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
4,longitude,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
5,payment_method,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
6,ridership,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
7,station_complex,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
8,station_complex_id,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
9,transfers,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."


Findings: The staged Subway parquet files exhibit a highly consistent schema across all quarterly extracts\. All expected temporal, station, fare, ridership, transfer, and geographic fields were present in every parquet file and retained consistent data types throughout the study period\. One unexpected field \(column0\) appeared only in the 2026 Q1 extract, indicating a minor schema anomaly that warrants further investigation\. Aside from this single field, no meaningful schema drift was observed across the 13 quarterly datasets\.

In [11]:
# ---------------------------------------------------------------------
# Inspect the unexpected column0 field if it exists
# ---------------------------------------------------------------------

subway_2026_q1_path = SUBWAY_PARQUET_DIR / "subway_2026_q1.parquet"

subway_2026_q1_columns = duckdb.sql(f"""
    DESCRIBE SELECT *
    FROM read_parquet('{subway_2026_q1_path}')
""").df()["column_name"].tolist()

if "column0" in subway_2026_q1_columns:
    column0_summary_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS total_rows,
            SUM(CASE WHEN column0 IS NULL THEN 1 ELSE 0 END) AS null_rows,
            COUNT(DISTINCT column0) AS distinct_values
        FROM read_parquet('{subway_2026_q1_path}')
    """).df()

    display(column0_summary_df)

else:
    print("column0 no longer exists in subway_2026_q1.parquet.")

column0 no longer exists in subway_2026_q1.parquet.


Findings: The unexpected column0 field in the 2026 Q1 parquet file appears to be an empty artifact rather than a meaningful source field\. All 5,250,000 rows are null and there are no distinct non\-null values\. This field can safely be ignored or dropped\. 

### Fixing the extra column in Q1\_2026

Let's drop that column now\.

In [12]:
# ---------------------------------------------------------------------
# Fix empty column0 artifact in Subway 2026 Q1 parquet
# ---------------------------------------------------------------------

subway_2026_q1_parquet_path = (
    SUBWAY_PARQUET_DIR / "subway_2026_q1.parquet"
)

subway_2026_q1_clean_path = (
    SUBWAY_PARQUET_DIR / "subway_2026_q1_clean.parquet"
)

duckdb.sql(f"""
    COPY (
        SELECT
            transit_timestamp,
            transit_mode,
            station_complex_id,
            station_complex,
            borough,
            payment_method,
            fare_class_category,
            ridership,
            transfers,
            latitude,
            longitude,
            georeference
        FROM read_parquet('{subway_2026_q1_parquet_path}')
    )
    TO '{subway_2026_q1_clean_path}'
    (FORMAT PARQUET);
""")

subway_2026_q1_clean_path.replace(subway_2026_q1_parquet_path)

print("Replaced subway_2026_q1.parquet without column0.")

Replaced subway_2026_q1.parquet without column0.


\.\.\. and let's recheck the schemas

In [13]:
# ---------------------------------------------------------------------
# 1.1.5.3 Schema QA for staged Subway parquet files
# ---------------------------------------------------------------------

subway_parquet_files = sorted(
    SUBWAY_PARQUET_DIR.glob("subway_*.parquet")
)

subway_schema_records = []

for path in subway_parquet_files:
    schema_df = duckdb.sql(f"""
        DESCRIBE SELECT *
        FROM read_parquet('{path}')
    """).df()

    for _, row in schema_df.iterrows():
        subway_schema_records.append({
            "file": path.name,
            "column_name": row["column_name"],
            "column_type": row["column_type"],
        })

subway_schema_df = pd.DataFrame(subway_schema_records)

subway_schema_df.head()

,file,column_name,column_type
0,subway_2023_q1.parquet,transit_timestamp,TIMESTAMP
1,subway_2023_q1.parquet,transit_mode,VARCHAR
2,subway_2023_q1.parquet,station_complex_id,VARCHAR
3,subway_2023_q1.parquet,station_complex,VARCHAR
4,subway_2023_q1.parquet,borough,VARCHAR


In [14]:
# ---------------------------------------------------------------------
# Compare Subway schemas across parquet files
# ---------------------------------------------------------------------

subway_schema_summary = (
    subway_schema_df
    .groupby(["column_name", "column_type"], dropna=False)
    .agg(
        num_files=("file", "nunique"),
        files=("file", lambda x: sorted(x.unique()))
    )
    .reset_index()
    .sort_values(["column_name", "column_type"])
)

subway_schema_summary

,column_name,column_type,num_files,files
0,borough,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
1,fare_class_category,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
2,georeference,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
3,latitude,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
4,longitude,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
5,payment_method,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
6,ridership,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
7,station_complex,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
8,station_complex_id,VARCHAR,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."
9,transfers,DOUBLE,13,"[subway_2023_q1.parquet, subway_2023_q2.parque..."


Findings\. Hooray \-\- it worked\! After removing the empty column0 artifact from the 2026 Q1 extract, all 13 Subway parquet files exhibited a fully consistent schema, with identical field names and data types across the entire study period and no remaining evidence of schema drift\.

### Schema consistency

Let's compare the observed schema against the expected Subway source fields\. This confirms that the core ridership, station, fare, time, and location fields survived staging\.

In [15]:
# ---------------------------------------------------------------------
# Confirm expected Subway columns are present
# ---------------------------------------------------------------------

expected_subway_columns = {
    "transit_timestamp",
    "transit_mode",
    "station_complex_id",
    "station_complex",
    "borough",
    "payment_method",
    "fare_class_category",
    "ridership",
    "transfers",
    "latitude",
    "longitude",
    "georeference",
}

observed_subway_columns = set(
    subway_schema_df["column_name"].unique()
)

missing_subway_columns = sorted(
    expected_subway_columns - observed_subway_columns
)

unexpected_subway_columns = sorted(
    observed_subway_columns - expected_subway_columns
)

print("Missing expected columns")
print(missing_subway_columns)

print("Unexpected columns")
print(unexpected_subway_columns)

Missing expected columns
[]
Unexpected columns
[]


Findings: All expected fields were present and no unexpected fields remained after resolving the temporary column0 artifact in the 2026 Q1 extract\.

Isolate any columns or data types that are not present across every parquet file\. If this table is empty, the staged Subway files have a consistent schema\.

In [16]:
# ---------------------------------------------------------------------
# Check for Subway schema drift across parquet files
# ---------------------------------------------------------------------

subway_num_parquet_files = len(subway_parquet_files)

subway_schema_drift_df = (
    subway_schema_summary[
        subway_schema_summary["num_files"] != subway_num_parquet_files
    ]
    .copy()
)

subway_schema_drift_df

,column_name,column_type,num_files,files


Findings: No schema drift was detected across the 13 quarterly parquet files\.

### Summary

The staged Subway dataset exhibited a highly consistent schema across all quarterly extracts\. Schema QA identified a single anomalous field \(column0\) in the 2026 Q1 parquet file; investigation confirmed that the field contained only null values and was an artifact of the staging process rather than a meaningful source attribute\. After removing the artifact and regenerating the affected parquet file, all quarterly extracts shared a fully consistent schema with no remaining evidence of schema drift and consistent data types throughout the study period\.

## 1\.1\.5\.4 Temporal QA

Let's validate the temporal characteristics of the staged Subway dataset\. Since the quarterly extracts were pulled directly from the MTA API using explicit date filters, our primary goal is to confirm that the observed timestamp ranges look reasonable and that the data behaves as expected for an hourly ridership dataset\. We will review timestamp coverage, hour\-level coverage, and timestamp granularity before moving on to field\-level quality checks\.

### Min / Max Timestamp Coverage

In [17]:
# ---------------------------------------------------------------------
# Check Subway timestamp coverage by parquet file
# ---------------------------------------------------------------------

subway_parquet_files = sorted(
    SUBWAY_PARQUET_DIR.glob("subway_*.parquet")
)

subway_temporal_records = []

for path in subway_parquet_files:
    temporal_summary = duckdb.sql(f"""
        SELECT
            MIN(transit_timestamp) AS min_transit_timestamp,
            MAX(transit_timestamp) AS max_transit_timestamp,
            COUNT(*) AS num_rows
        FROM read_parquet('{path}')
    """).df()

    subway_temporal_records.append({
        "file": path.name,
        "min_transit_timestamp": temporal_summary.loc[0, "min_transit_timestamp"],
        "max_transit_timestamp": temporal_summary.loc[0, "max_transit_timestamp"],
        "num_rows": temporal_summary.loc[0, "num_rows"],
    })

subway_temporal_df = pd.DataFrame(subway_temporal_records)

subway_temporal_df

,file,min_transit_timestamp,max_transit_timestamp,num_rows
0,subway_2023_q1.parquet,2023-01-01,2023-03-31 23:00:00,6268873
1,subway_2023_q2.parquet,2023-04-01,2023-06-30 23:00:00,6413089
2,subway_2023_q3.parquet,2023-07-01,2023-09-30 23:00:00,6375366
3,subway_2023_q4.parquet,2023-10-01,2023-12-31 23:00:00,6528085
4,subway_2024_q1.parquet,2024-01-01,2024-03-31 23:00:00,6445692
5,subway_2024_q2.parquet,2024-04-01,2024-06-30 23:00:00,6558471
6,subway_2024_q3.parquet,2024-07-01,2024-09-30 23:00:00,6692309
7,subway_2024_q4.parquet,2024-10-01,2024-12-31 23:00:00,7316041
8,subway_2025_q1.parquet,2025-01-01,2025-03-31 23:00:00,7271588
9,subway_2025_q2.parquet,2025-04-01,2025-06-30 23:00:00,8095749


Findings: The temporal coverage of the staged Subway extracts aligns exactly with the intended quarterly boundaries used during data extraction\. Each parquet file begins at the first hour of its quarter and ends at the final hour of the final day, with no evidence of temporal gaps, overlap, or truncation\. This provides strong validation that the paginated API extraction captured the complete observation period for each quarter\.

## Distinct Hour Coverage

Next, we check the distinct hours represented in each parquet file\. Since this is an hourly ridership dataset, we expect observations to span all 24 hours of the day\.

In [18]:
# ---------------------------------------------------------------------
# Check distinct Subway hours by parquet file
# ---------------------------------------------------------------------

subway_hour_records = []

for path in subway_parquet_files:
    hour_df = duckdb.sql(f"""
        SELECT
            EXTRACT(HOUR FROM transit_timestamp) AS hour,
            COUNT(*) AS num_rows
        FROM read_parquet('{path}')
        GROUP BY
            hour
        ORDER BY
            hour
    """).df()

    hour_df["file"] = path.name

    subway_hour_records.append(hour_df)

subway_hour_df = pd.concat(
    subway_hour_records,
    ignore_index=True
)

subway_hour_coverage_df = (
    subway_hour_df
    .groupby("file")
    .agg(
        num_distinct_hours=("hour", "nunique"),
        min_hour=("hour", "min"),
        max_hour=("hour", "max"),
    )
    .reset_index()
)

subway_hour_coverage_df

,file,num_distinct_hours,min_hour,max_hour
0,subway_2023_q1.parquet,24,0,23
1,subway_2023_q2.parquet,24,0,23
2,subway_2023_q3.parquet,24,0,23
3,subway_2023_q4.parquet,24,0,23
4,subway_2024_q1.parquet,24,0,23
5,subway_2024_q2.parquet,24,0,23
6,subway_2024_q3.parquet,24,0,23
7,subway_2024_q4.parquet,24,0,23
8,subway_2025_q1.parquet,24,0,23
9,subway_2025_q2.parquet,24,0,23


Findings: All quarterly Subway extracts contain observations for every hour of the day, with complete coverage from hour 0 through hour 23\. This confirms that the dataset represents a full hourly time series and that no hours were systematically omitted during extraction or staging\.

### Minute\-Level Granularity

Finally, let's inspect the minute values present in the timestamps\. This helps confirm whether the dataset is truly hourly or whether the timestamps contain finer\-grained intervals that may need to be considered during harmonization\.

In [19]:
# ---------------------------------------------------------------------
# Check Subway timestamp minute values
# ---------------------------------------------------------------------

subway_minute_records = []

for path in subway_parquet_files:
    minute_df = duckdb.sql(f"""
        SELECT
            EXTRACT(MINUTE FROM transit_timestamp) AS minute,
            COUNT(*) AS num_rows
        FROM read_parquet('{path}')
        GROUP BY
            minute
        ORDER BY
            minute
    """).df()

    minute_df["file"] = path.name

    subway_minute_records.append(minute_df)

subway_minute_df = pd.concat(
    subway_minute_records,
    ignore_index=True
)

subway_minute_df.sort_values(
    ["file", "minute"]
)

,minute,num_rows,file
0,0,6268873,subway_2023_q1.parquet
1,0,6413089,subway_2023_q2.parquet
2,0,6375366,subway_2023_q3.parquet
3,0,6528085,subway_2023_q4.parquet
4,0,6445692,subway_2024_q1.parquet
5,0,6558471,subway_2024_q2.parquet
6,0,6692309,subway_2024_q3.parquet
7,0,7316041,subway_2024_q4.parquet
8,0,7271588,subway_2025_q1.parquet
9,0,8095749,subway_2025_q2.parquet


Findings: All observations occur exactly at minute 0 of the hour, confirming that the dataset is truly hourly rather than a finer\-grained event stream\. No evidence of sub\-hourly timestamps was observed, which simplifies downstream temporal alignment with other mobility datasets\.

## 1\.1\.5\.5 Missingness and Duplication QA

Let's check for missing values and duplicate records in the staged Subway data\. At this point, we are not trying to transform anything yet — we just want to confirm that the core timestamp, station, fare, ridership, and location fields are usable before moving into standardization\.

### Missingness checks

First, we summarize missingness across the key Subway fields\.

In [20]:
# ---------------------------------------------------------------------
# Check Subway missingness across key fields
# ---------------------------------------------------------------------

subway_key_fields = [
    "transit_timestamp",
    "transit_mode",
    "station_complex_id",
    "station_complex",
    "borough",
    "payment_method",
    "fare_class_category",
    "ridership",
    "transfers",
    "latitude",
    "longitude",
    "georeference",
]

subway_missingness_records = []

for path in subway_parquet_files:
    select_expressions = []

    for field in subway_key_fields:
        select_expressions.append(
            f"SUM(CASE WHEN {field} IS NULL THEN 1 ELSE 0 END) AS {field}_null_count"
        )

    missingness_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS num_rows,
            {", ".join(select_expressions)}
        FROM read_parquet('{path}')
    """).df()

    for field in subway_key_fields:
        null_count = missingness_df.loc[0, f"{field}_null_count"]
        num_rows = missingness_df.loc[0, "num_rows"]

        subway_missingness_records.append({
            "file": path.name,
            "field": field,
            "num_rows": num_rows,
            "null_count": null_count,
            "null_pct": null_count / num_rows if num_rows > 0 else None,
        })

subway_missingness_df = pd.DataFrame(subway_missingness_records)

subway_missingness_summary = (
    subway_missingness_df
    .groupby("field", dropna=False)
    .agg(
        total_rows=("num_rows", "sum"),
        total_nulls=("null_count", "sum"),
        max_file_null_pct=("null_pct", "max"),
    )
    .reset_index()
)

subway_missingness_summary["total_null_pct"] = (
    subway_missingness_summary["total_nulls"] /
    subway_missingness_summary["total_rows"]
)

subway_missingness_summary.sort_values(
    "total_null_pct",
    ascending=False
)

,field,total_rows,total_nulls,max_file_null_pct,total_null_pct
0,borough,88319707,0.0,0.0,0.0
1,fare_class_category,88319707,0.0,0.0,0.0
2,georeference,88319707,0.0,0.0,0.0
3,latitude,88319707,0.0,0.0,0.0
4,longitude,88319707,0.0,0.0,0.0
5,payment_method,88319707,0.0,0.0,0.0
6,ridership,88319707,0.0,0.0,0.0
7,station_complex,88319707,0.0,0.0,0.0
8,station_complex_id,88319707,0.0,0.0,0.0
9,transfers,88319707,0.0,0.0,0.0


Findings: No missing values were observed across any of the key temporal, station, fare, ridership, transfer, or geographic fields\. All 88\.3 million records contained complete information for the attributes required by downstream temporal, spatial, and multimodal integration workflows\.

### Duplication validation

Let's validate the candidate observation key\. For Subway, each observation should be unique by timestamp, mode, station complex, payment method, and fare class category\.

In [21]:
# ---------------------------------------------------------------------
# Check Subway candidate observation key duplication
# ---------------------------------------------------------------------

# Subway observations are not unique by station and timestamp alone.
# Payment method and fare class are part of the native reporting grain.
subway_candidate_key_fields = [
    "transit_timestamp",
    "transit_mode",
    "station_complex_id",
    "payment_method",
    "fare_class_category",
]

subway_key_duplicate_records = []

for path in subway_parquet_files:
    key_fields_sql = ", ".join(subway_candidate_key_fields)

    # Group at the candidate source grain and look for keys with more than one row.
    # Any duplicate key groups here would point to duplicate download or staging records.
    duplicate_key_df = duckdb.sql(f"""
        SELECT
            COUNT(*) AS duplicate_key_groups,
            SUM(num_rows) AS rows_in_duplicate_key_groups
        FROM (
            SELECT
                {key_fields_sql},
                COUNT(*) AS num_rows
            FROM read_parquet('{path}')
            GROUP BY
                {key_fields_sql}
            HAVING
                COUNT(*) > 1
        )
    """).df()

    total_rows = duckdb.sql(f"""
        SELECT COUNT(*) AS total_rows
        FROM read_parquet('{path}')
    """).fetchone()[0]

    duplicate_key_groups = duplicate_key_df.loc[0, "duplicate_key_groups"]
    rows_in_duplicate_key_groups = duplicate_key_df.loc[0, "rows_in_duplicate_key_groups"]

    duplicate_key_groups = 0 if pd.isna(duplicate_key_groups) else duplicate_key_groups
    rows_in_duplicate_key_groups = 0 if pd.isna(rows_in_duplicate_key_groups) else rows_in_duplicate_key_groups

    subway_key_duplicate_records.append({
        "file": path.name,
        "total_rows": total_rows,
        "duplicate_key_groups": duplicate_key_groups,
        "rows_in_duplicate_key_groups": rows_in_duplicate_key_groups,
        "duplicate_key_row_pct": (
            rows_in_duplicate_key_groups / total_rows
            if total_rows > 0 else None
        ),
    })

subway_key_duplicate_df = pd.DataFrame(
    subway_key_duplicate_records
)

subway_key_duplicate_df

,file,total_rows,duplicate_key_groups,rows_in_duplicate_key_groups,duplicate_key_row_pct
0,subway_2023_q1.parquet,6268873,0,0,0.0
1,subway_2023_q2.parquet,6413089,0,0,0.0
2,subway_2023_q3.parquet,6375366,0,0,0.0
3,subway_2023_q4.parquet,6528085,0,0,0.0
4,subway_2024_q1.parquet,6445692,0,0,0.0
5,subway_2024_q2.parquet,6558471,0,0,0.0
6,subway_2024_q3.parquet,6692309,0,0,0.0
7,subway_2024_q4.parquet,7316041,0,0,0.0
8,subway_2025_q1.parquet,7271588,0,0,0.0
9,subway_2025_q2.parquet,8095749,0,0,0.0


Findings: No duplicate observation keys were detected in any quarterly Subway extract\. Each combination of timestamp, transit mode, station complex, payment method, and fare class category appeared exactly once, providing strong evidence that the paginated API extraction and parquet staging processes did not introduce duplicate observations\.

## 1\.1\.5\.6 Metric QA

This QA focuses on whether the Subway mobility metrics behave sensibly as numeric measures\. We will check for impossible values, review the distribution of ridership and transfers, and sanity\-check the largest ridership observations\.

### Metric Health and Distribution

Let's start by checking for negative values, zero values, and basic min/max ranges\.

In [22]:

# ---------------------------------------------------------------------
# Check Subway metric validity
# ---------------------------------------------------------------------

subway_parquet_glob = str(
    SUBWAY_PARQUET_DIR / "subway_*.parquet"
)

subway_metric_validity_df = duckdb.sql(f"""
    SELECT
        COUNT(*) AS total_rows,

        SUM(CASE WHEN ridership < 0 THEN 1 ELSE 0 END) AS ridership_negatives,
        SUM(CASE WHEN transfers < 0 THEN 1 ELSE 0 END) AS transfers_negatives,

        SUM(CASE WHEN ridership = 0 THEN 1 ELSE 0 END) AS ridership_zeros,
        SUM(CASE WHEN transfers = 0 THEN 1 ELSE 0 END) AS transfers_zeros,

        MIN(ridership) AS ridership_min,
        MAX(ridership) AS ridership_max,

        MIN(transfers) AS transfers_min,
        MAX(transfers) AS transfers_max

    FROM read_parquet('{subway_parquet_glob}')
""").df()

subway_metric_validity_df

,total_rows,ridership_negatives,transfers_negatives,ridership_zeros,transfers_zeros,ridership_min,ridership_max,transfers_min,transfers_max
0,88319707,0.0,0.0,0.0,63603698.0,1.0,21602.0,0.0,2257.0


Findings: No negative ridership or transfer values were observed, indicating that the core mobility metrics are internally consistent and free from obvious data\-quality issues\. Ridership values ranged from 1 to 21,602 riders per station\-hour, while transfers ranged from 0 to 2,257 per station\-hour\. Transfer activity was zero in approximately 72% of observations, which is plausible given that many station\-hour combinations may not generate measurable transfer activity\.

Next, we examine the distribution of ridership and transfers\. Given the size of the staged Subway dataset \(more than 88 million observations\), we use approximate quantiles to efficiently characterize the typical range of activity as well as the extreme upper tail\. This provides a practical summary of metric behavior without requiring memory\-intensive exact percentile calculations\.

In [23]:
# ---------------------------------------------------------------------
# Check Subway metric distributions
# ---------------------------------------------------------------------

subway_parquet_file_paths = [
    str(path)
    for path in subway_parquet_files
]

subway_metric_distribution_df = duckdb.sql(f"""
    SELECT
        approx_quantile(ridership, 0.50) AS ridership_p50,
        approx_quantile(ridership, 0.90) AS ridership_p90,
        approx_quantile(ridership, 0.95) AS ridership_p95,
        approx_quantile(ridership, 0.99) AS ridership_p99,
        approx_quantile(ridership, 0.999) AS ridership_p999,
        MAX(ridership) AS ridership_max,

        approx_quantile(transfers, 0.50) AS transfers_p50,
        approx_quantile(transfers, 0.90) AS transfers_p90,
        approx_quantile(transfers, 0.95) AS transfers_p95,
        approx_quantile(transfers, 0.99) AS transfers_p99,
        approx_quantile(transfers, 0.999) AS transfers_p999,
        MAX(transfers) AS transfers_max

    FROM read_parquet({subway_parquet_file_paths})
""").df()

subway_metric_distribution_df

,ridership_p50,ridership_p90,ridership_p95,ridership_p99,ridership_p999,ridership_max,transfers_p50,transfers_p90,transfers_p95,transfers_p99,transfers_p999,transfers_max
0,9.197694,84.542611,171.806377,632.514715,2665.737346,21602.0,0.0,3.368162,7.736037,33.237968,176.940183,2257.0


Findings: Subway ridership is highly right\-skewed, with a median of approximately 9 riders per station\-hour compared to a maximum observed value of 21,602 riders\. Even the 99th percentile remained below 172 riders, indicating that the largest ridership observations represent a relatively small number of exceptionally high\-volume station\-hours\. Transfer activity exhibited a similar pattern, with a median of 0 transfers and most observations concentrated at very low values, while the upper tail reflects a limited number of major transfer hubs and peak travel periods\.

### Top Ridership Validation

Finally, let's inspect the largest ridership observations directly\. If the biggest values are coming from plausible high\-volume stations, fare classes, and timestamps, that gives us more confidence that the upper tail is real rather than a staging issue\.

In [24]:
# ---------------------------------------------------------------------
# Review largest Subway ridership observations
# ---------------------------------------------------------------------

subway_top_ridership_df = duckdb.sql(f"""
    SELECT
        transit_timestamp,
        station_complex_id,
        station_complex,
        borough,
        payment_method,
        fare_class_category,
        ridership,
        transfers

    FROM read_parquet('{subway_parquet_glob}')

    ORDER BY
        ridership DESC

    LIMIT 25
""").df()

subway_top_ridership_df

,transit_timestamp,station_complex_id,station_complex,borough,payment_method,fare_class_category,ridership,transfers
0,2025-12-10 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,21602.0,61.0
1,2026-02-03 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,21156.0,75.0
2,2026-01-13 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20873.0,81.0
3,2025-12-09 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20828.0,62.0
4,2026-03-03 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20755.0,104.0
5,2026-02-10 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20666.0,84.0
6,2026-01-28 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20484.0,79.0
7,2026-01-20 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20471.0,95.0
8,2026-02-04 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20454.0,82.0
9,2026-03-18 17:00:00,610,"Grand Central-42 St (4,5,6,7,S)",Manhattan,omny,OMNY - Full Fare,20366.0,98.0


Findings: The largest ridership observations appear reasonable and provide additional confidence in the validity of the data\. All top 25 records correspond to the Grand Central–42 St station complex during the weekday evening commute hour \(5 PM\), with ridership ranging from approximately 19,000 to 21,600 riders\. The concentration of these peak observations at one of the busiest transit hubs in the system during a known rush\-hour period suggests that the extreme upper tail of the ridership distribution reflects genuine transit demand rather than data\-quality issues or extraction errors\.

## 1\.1\.5\.7 Geography and Categorical Coverage QA

Let's close the Subway source QA by checking the fields that matter most for downstream mobility analysis\. We want to confirm that the dataset has usable station coverage, usable geography, sensible borough\-level distribution, and the operational categories needed to interpret ridership before aggregation\.

First, let's confirm basic station coverage\.

In [25]:
# ---------------------------------------------------------------------
# Check Subway station coverage
# ---------------------------------------------------------------------

subway_station_coverage_df = duckdb.sql(f"""
    SELECT
        COUNT(DISTINCT station_complex_id) AS distinct_station_complex_ids,
        COUNT(DISTINCT station_complex) AS distinct_station_complex_names
    FROM read_parquet('{subway_parquet_glob}')
""").df()

subway_station_coverage_df

,distinct_station_complex_ids,distinct_station_complex_names
0,428,466


Findings: The staged Subway dataset contains 428 unique station complex identifiers representing 466 station complex names\. This level of coverage is consistent with a system\-wide extract and suggests that the dataset captures activity across the full NYC Subway network rather than a subset of stations\.

Let's summarize ridership by borough\. This gives us a quick read on geographic coverage and where Subway activity is concentrated\.

In [26]:
# ---------------------------------------------------------------------
# Check Subway borough distribution
# ---------------------------------------------------------------------

subway_borough_distribution_df = duckdb.sql(f"""
    SELECT
        borough,
        COUNT(*) AS num_rows,
        SUM(ridership) AS total_ridership,
        SUM(transfers) AS total_transfers
    FROM read_parquet('{subway_parquet_glob}')
    GROUP BY
        borough
    ORDER BY
        total_ridership DESC
""").df()

subway_borough_distribution_df

,borough,num_rows,total_ridership,total_transfers
0,Manhattan,27319209,2.223312e+09,51928156.0
1,Brooklyn,31327314,8.887083e+08,41000484.0
2,Queens,15451827,5.910614e+08,63388031.0
3,Bronx,13872559,2.754465e+08,18451312.0
4,Staten Island,348798,6.909930e+06,1764448.0


Findings: Subway activity is concentrated in Manhattan, which accounts for approximately 2\.2 billion rider entries and more than 51 million transfers during the study period\. Brooklyn, Queens, and the Bronx also contribute substantial ridership volume, while Staten Island represents only a small share of overall activity\. The distribution aligns with expectations given the density of Manhattan's transit network and its role as the primary employment and transfer hub within the NYC transportation system\.

Finally, let's review the operational categories that segment ridership\. These fields may affect how we aggregate Subway activity later\.

In [27]:
# ---------------------------------------------------------------------
# Check Subway operational category coverage
# ---------------------------------------------------------------------

subway_operational_category_records = []

for field in [
    "transit_mode",
    "payment_method",
    "fare_class_category",
]:
    category_df = duckdb.sql(f"""
        SELECT
            '{field}' AS field,
            {field} AS category,
            COUNT(*) AS num_rows,
            SUM(ridership) AS total_ridership,
            SUM(transfers) AS total_transfers
        FROM read_parquet('{subway_parquet_glob}')
        GROUP BY
            {field}
        ORDER BY
            num_rows DESC
    """).df()

    subway_operational_category_records.append(category_df)

subway_operational_category_df = pd.concat(
    subway_operational_category_records,
    ignore_index=True
)

subway_operational_category_df

,field,category,num_rows,total_ridership,total_transfers
0,transit_mode,subway,87633835,3.968767e+09,172674086.0
1,transit_mode,staten_island_railway,348797,6.909928e+06,1764448.0
2,transit_mode,tram,337075,9.760728e+06,2093897.0
3,payment_method,metrocard,59981386,1.425193e+09,59757194.0
4,payment_method,omny,28338321,2.560245e+09,116775237.0
5,fare_class_category,OMNY - Full Fare,11640588,2.307522e+09,97159115.0
6,fare_class_category,Metrocard - Other,10377677,1.420750e+08,2570500.0
7,fare_class_category,Metrocard - Unlimited 30-Day,10087706,2.688777e+08,10.0
8,fare_class_category,Metrocard - Unlimited 7-Day,10067360,2.759210e+08,29.0
9,fare_class_category,Metrocard - Full Fare,10050643,4.602284e+08,34799024.0


Findings: The dataset is overwhelmingly composed of Subway observations, with comparatively small contributions from the Staten Island Railway and Roosevelt Island Tram\. Payment activity reflects the ongoing transition from MetroCard to OMNY, with OMNY accounting for approximately 2\.6 billion rider entries compared to 1\.4 billion for MetroCard\. Across fare products, full\-fare OMNY riders represent the largest segment by a substantial margin, while student, senior, disability, fair\-fare, and unlimited\-pass categories are all well represented, indicating broad coverage of the MTA fare ecosystem\.

In [28]:
# ---------------------------------------------------------------------
# Cleanup
# ---------------------------------------------------------------------

import gc

gc.collect()

40

## Close

The Subway source staging process completed successfully across 13 quarterly extracts spanning Q1 2023 through Q1 2026\. Approximately 88\.3 million observations were extracted, validated, and converted into a compact parquet representation suitable for downstream analysis\. Schema QA identified a single staging artifact in the 2026 Q1 extract, which was investigated, corrected, and revalidated\. Temporal QA confirmed complete quarterly and hourly coverage, missingness QA found no null values in the key analytical fields, metric QA revealed plausible ridership and transfer distributions, and geography/category QA confirmed broad coverage across stations, boroughs, fare products, and payment methods\. The staged Subway dataset is now ready for harmonization, Taxi Zone alignment, and multimodal integration with the remaining transportation systems\.

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4a322346-8e1e-4650-8cef-fe9b767d96fb' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>